# 3T WALRUS versus HLSVD + L2

Paired evaluation on identical simulations for all validation subjects. Errors are calculated per spectrum, averaged within each subject, and summarized as mean ± sample SD across subjects.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = Path('.').resolve()

DEVICE = 'cuda:0'
MODEL_DIR = 'models/3T_Final'
CHECKPOINT = 'model_last.pt'
N_SPECTRA_PER_SUBJECT = 30_000
SIMULATION_BATCH_SIZE = 512
HSVD_BATCH_SIZE = 128
SEED = 20260915
RESULT_PATH = 'results/HLSVD_L2_Comparison/3T_validation_clean_metabolite.h5'
OVERWRITE = False  # False resumes completed/partial subject batches
SAVE_PDF = None  # e.g. PROJECT_ROOT / 'results/HLSVD_L2_Comparison/comparison.pdf'

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.hlsvd_l2_comparison import run_evaluation

result_path = run_evaluation(
    project_root=PROJECT_ROOT,
    model_dir=MODEL_DIR,
    checkpoint=CHECKPOINT,
    output_path=RESULT_PATH,
    n_spectra_per_subject=N_SPECTRA_PER_SUBJECT,
    simulation_batch_size=SIMULATION_BATCH_SIZE,
    hsvd_batch_size=HSVD_BATCH_SIZE,
    seed=SEED,
    device=DEVICE,
    overwrite=OVERWRITE,
)
result_path

In [ ]:
from walinet.paper_figures.hlsvd_l2_comparison import results_dataframe

subject_results = results_dataframe(result_path)
summary = (
    subject_results.groupby(['metric', 'method'])['mean']
    .agg(mean_over_subjects='mean', std_over_subjects='std', n_subjects='count')
)
summary

In [ ]:
from walinet.paper_figures.hlsvd_l2_comparison import plot_comparison

figure, subject_results = plot_comparison(result_path)
if SAVE_PDF is not None:
    save_path = Path(SAVE_PDF).expanduser()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(save_path, format='pdf', bbox_inches='tight')
    print(f'Saved: {save_path.resolve()}')
figure